# SupplyMind AI — Baseline Model

This notebook trains the first defensible baseline using Logistic Regression.
It imports the shared production preprocessing, splitting, training, and
evaluation modules instead of duplicating logic.

In [ ]:
# -------------------
# Imports
# -------------------

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression

from supplymind.features.predictions.application.dataset import (
    load_tabular_dataset,
    normalize_column_names,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
from supplymind.features.predictions.ml.cleaning import clean_shipment_data
from supplymind.features.predictions.ml.evaluation import (
    evaluate_binary_classifier,
)
from supplymind.features.predictions.ml.features import add_datetime_features
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.splitting import (
    temporal_train_validation_test_split,
)
from supplymind.features.predictions.ml.training import train_pipeline

In [ ]:
# -------------------
# Configuration
# -------------------

DATASET_PATH = Path("../data/raw/syndelay/YOUR_DATASET_FILE.csv")
TARGET_COLUMN = "REPLACE_WITH_CONFIRMED_TARGET"
TIMESTAMP_COLUMN = "REPLACE_WITH_CONFIRMED_TIMESTAMP"

NUMERICAL_FEATURES = [
    # Add confirmed numerical features.
]

CATEGORICAL_FEATURES = [
    # Add confirmed categorical features.
]

ARTIFACT_DIRECTORY = Path("../models/baseline_logistic_regression")

In [ ]:
# -------------------
# Data preparation
# -------------------

df = load_tabular_dataset(DATASET_PATH)
df = normalize_column_names(df)
df = clean_shipment_data(df)
df = add_datetime_features(df, [TIMESTAMP_COLUMN])

split = temporal_train_validation_test_split(
    df,
    TIMESTAMP_COLUMN,
)

feature_columns = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

X_train = split.train[feature_columns]
y_train = split.train[TARGET_COLUMN]

X_validation = split.validation[feature_columns]
y_validation = split.validation[TARGET_COLUMN]

X_test = split.test[feature_columns]
y_test = split.test[TARGET_COLUMN]

In [ ]:
# -------------------
# Baseline pipeline
# -------------------

preprocessor = build_preprocessor(
    numerical_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    scale_numerical=True,
)

baseline_estimator = LogisticRegression(
    max_iter=1_000,
    class_weight="balanced",
    random_state=42,
)

baseline_model = train_pipeline(
    preprocessor=preprocessor,
    estimator=baseline_estimator,
    features=X_train,
    target=y_train,
)

In [ ]:
# -------------------
# Validation evaluation
# -------------------

validation_metrics = evaluate_binary_classifier(
    baseline_model,
    X_validation,
    y_validation,
    threshold=0.50,
)

validation_metrics.to_dict()

## Important

Do not evaluate on the test set until model comparison and model selection
are complete. The untouched test set is reserved for one final evaluation
of the selected champion.

In [ ]:
# -------------------
# Save baseline artifact
# -------------------

baseline_metadata = {
    "model_name": "logistic_regression",
    "model_version": "0.1.0-baseline",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "target_column": TARGET_COLUMN,
    "timestamp_column": TIMESTAMP_COLUMN,
    "feature_columns": feature_columns,
    "validation_metrics": validation_metrics.to_dict(),
    "decision_threshold": 0.50,
    "dataset_name": "SynDelay",
    "artifact_status": "baseline_not_champion",
}

save_model_artifact(
    baseline_model,
    baseline_metadata,
    ARTIFACT_DIRECTORY,
)

print(f"Saved baseline artifact to: {ARTIFACT_DIRECTORY}")